## Notebook setup

This notebook follows the same sequence as the Learning Path pages: create a reference model, export with the ExecuTorch VGF backend, validate with `executor_runner`, inspect artifacts in Model Explorer, and optionally inspect the Tensor Operator Set Architecture (TOSA) middle step.

Before running this notebook, complete the environment setup from the Learning Path. 

After completing setup, launch Jupyter Lab from the `preparing-models-for-nt` directory:

```bash
jupyter lab
```

For the kernel, select the virtual environment that you made using the setup script.


## Create a reference model

Start with a small `AddSigmoid` model so the export and validation flow is easy to inspect.


In [ ]:
import torch


class AddSigmoid(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return self.sigmoid(x + y)

example_inputs = (torch.ones(1, 1, 1, 1), torch.ones(1, 1, 1, 1))

model = AddSigmoid().eval()
exported_model = torch.export.export(model, example_inputs)
graph_module = exported_model.module(check_guards=False)

_ = graph_module.print_readable()


torch.export.save(exported_model, "add_sigmoid.pt2")
print("Wrote: add_sigmoid.pt2")

## Export with the ExecuTorch VGF backend

VGF is the file format used by Arm neural technology with ML Extensions for Vulkan. Through ExecuTorch, you can lower directly from a PyTorch model to a VGF model using the direct VGF backend path first. 

An important note is that TOSA inspection is available later as an optional debugging step to view operations of your model. 


In [ ]:
from executorch.backends.arm.vgf import VgfCompileSpec, VgfPartitioner
from executorch.exir import (
    EdgeCompileConfig,
    ExecutorchBackendConfig,
    to_edge_transform_and_lower,
)
from executorch.extension.export_util.utils import save_pte_program
import os
import torch

os.makedirs("executorch-model", exist_ok=True)

exported_model = torch.export.load("add_sigmoid.pt2")

# VGF lowering
compile_spec = VgfCompileSpec()    
compile_spec.dump_intermediate_artifacts_to("executorch-model")   # default FP compile spec
partitioner = VgfPartitioner(compile_spec)

edge_pm = to_edge_transform_and_lower(
    exported_model,
    partitioner=[partitioner],
    compile_config=EdgeCompileConfig(_check_ir_validity=False),
)

et_pm = edge_pm.to_executorch(
    config=ExecutorchBackendConfig(extract_delegate_segments=False)
)

# Save .pte
cwd_dir = os.getcwd()
pte_base_name = "as-vgf"
out_name = pte_base_name + ".pte"
save_pte_program(et_pm, out_name)
print("Wrote:", os.path.abspath(out_name))
pte_path = os.path.join(cwd_dir, out_name)

graph_module = exported_model.module(check_guards=False)
_ = graph_module.print_readable() # for visibility 

Now you'll use the Model Explorer to visualize the models. 

Model Explorer becomes useful later as you use more complex models.

## Start Model Explorer

If you haven't already, make sure the Model Explorer and the adapters are installed:

```bash
pip install torch ai-edge-model-explorer
pip install pte-adapter-model-explorer
pip install tosa-adapter-model-explorer
pip install vgf-adapter-model-explorer
```

Run the following command in a terminal to start Model Explorer with the PTE, TOSA, and VGF adapters:

```bash
model-explorer --extensions=pte_adapter_model_explorer,tosa_adapter_model_explorer,vgf_adapter_model_explorer
```

Start by opening the `.vgf` or `.pte` artifacts generated by the ExecuTorch VGF backend. If you run the optional TOSA cells later, you can compare the intermediate representation with the deployable output.


## Build and run runtime validation

Build `executor_runner`, then run the generated `.pte` file using the VKML helper script.


In [ ]:
%%bash
# Ensure the Vulkan environment variables and ML SDK for Vulkan components are available on $PATH
source repo/executorch/examples/arm/arm-scratch/setup_path.sh
cd repo/executorch/examples/arm

# Compiled programs will appear in the executorch/cmake-out directory.
# Build example executor runner application to examples/arm/vgf_minimal_example
cmake \
  -DCMAKE_INSTALL_PREFIX=cmake-out \
  -DCMAKE_BUILD_TYPE=Debug \
  -DEXECUTORCH_BUILD_EXTENSION_DATA_LOADER=ON \
  -DEXECUTORCH_BUILD_EXTENSION_MODULE=ON \
  -DEXECUTORCH_BUILD_EXTENSION_NAMED_DATA_MAP=ON \
  -DEXECUTORCH_BUILD_EXTENSION_FLAT_TENSOR=ON \
  -DEXECUTORCH_BUILD_EXTENSION_TENSOR=ON \
  -DEXECUTORCH_BUILD_KERNELS_QUANTIZED=ON \
  -DEXECUTORCH_BUILD_XNNPACK=OFF \
  -DEXECUTORCH_BUILD_VULKAN=ON \
  -DEXECUTORCH_BUILD_VGF=ON \
  -DEXECUTORCH_ENABLE_LOGGING=ON \
  -DPYTHON_EXECUTABLE=python \
  -B../../cmake-out-vkml ../..

cmake --build ../../cmake-out-vkml --target executor_runner

Now that you've built the runtime, use the `run_vkml.sh` script to run the model with it:

In [ ]:
import subprocess

cwd_dir = os.getcwd()


# Setup paths
et_dir = os.path.join(cwd_dir)

script_dir = os.path.join(et_dir, "repo", "executorch" ,"backends", "arm", "scripts")
et_dir = os.path.join(et_dir, "repo" , "executorch")
pte_path = os.path.abspath("as-vgf.pte")

args = f"--model={pte_path}"
subprocess.run(os.path.join(script_dir, "run_vkml.sh") + " " + args, shell=True, cwd=et_dir, check=True)

## (Optional) Inspect TOSA artifacts

Run these cells only when you want to inspect or compare the intermediate TOSA route.


In [ ]:
# Optional deeper step: dump TOSA for the model FP variant

from pathlib import Path

import torch
from executorch.exir import to_edge_transform_and_lower, EdgeCompileConfig
from executorch.backends.arm.tosa import TosaSpecification
from executorch.backends.arm.tosa.compile_spec import TosaCompileSpec
from executorch.backends.arm.util._factory import create_partitioner


BASE_DUMP = Path("tosa-dump")

def dump_tosa(ep, profile_str: str, label: str):
    BASE_DUMP.mkdir(parents=True, exist_ok=True)

    tosa_spec = TosaSpecification.create_from_string(profile_str)
    compile_spec = TosaCompileSpec(tosa_spec)

    # Keep intermediate artifacts in one folder for inspection
    compile_spec.dump_intermediate_artifacts_to(str(BASE_DUMP))

    partitioner = create_partitioner(compile_spec)

    _ = to_edge_transform_and_lower(
        ep,
        partitioner=[partitioner],
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
    )

    tosa_files = list(BASE_DUMP.rglob("*.tosa"))

    print(f"\n{label}")
    print(f"  Profile: {profile_str}")
    print(f"  Dump dir: {BASE_DUMP.resolve()}")
    print(f"  Total .tosa files so far: {len(tosa_files)}")

exported_model = torch.export.load("add_sigmoid.pt2")
dump_tosa(exported_model, "TOSA-1.0+FP", "AddSigmoidModel (float)")



After validating your model, you can use the Model Converter to turn the `.tosa` model into a `.vgf` file, or use the ExecuTorch path as you did earlier. 

In [ ]:
# Optional: convert the dumped .tosa model to a .vgf
import pathlib
tosa_path = pathlib.Path("./tosa-dump/output_tag0_TOSA-1.0+FP.tosa")
vgf_path = pathlib.Path("./executorch-model/model-from-tosa.vgf")

subprocess.run(
    [
        "model_converter",
        "--input",str(tosa_path),
        "--output",str(vgf_path),
    ],
    check=True,
)

## Choose the right Arm tooling path

Use this notebook when you need direct control over VGF export, TOSA inspection, and runtime validation. For higher-level training and export workflows, continue with Model Gym. For packaged Vulkan validation, use the Scenario Runner or the Simple Tensor and Data Graph sample linked from the Learning Path.
